# A0 原表达与 A1 PCA

独立 kernel、单卡顺序运行两组；默认 3 个切片 × 5 个 seed。完整协议见 [README](README.md#a0a5-欧氏对照实验)。本轮是探索性表示实验，不使用隐藏切片选参。


In [1]:
# 必须在导入 torch 前运行；切换显卡或更新代码后重启 kernel。
import os
import sys
GPU_ID = "7"  # 调度器管理可见卡时设为 None。
if "torch" in sys.modules:
    raise RuntimeError("请重启独立 kernel，并从第一格运行。")
if GPU_ID is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES", "scheduler/default"))


CUDA_VISIBLE_DEVICES = 7


In [2]:
from pathlib import Path
from copy import deepcopy
PROJECT = next((candidate for parent in (Path.cwd(), *Path.cwd().parents)
                for candidate in (parent, parent / "MMDeepSpatial")
                if (candidate / "mmdeepspatial/euclidean.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("请在 MMDeepSpatial 或其父目录打开 notebook。")
sys.path.insert(0, str(PROJECT))
import pandas as pd
from mmdeepspatial.euclidean import EUCLIDEAN_CONFIG, run_arm, compare_arms
from mmdeepspatial.data import input_paths
print("Project:", PROJECT)


Project: /3250608056/code/DeepSpatial/MMDeepSpatial


## 统一配置

三个 notebook 的科学配置须一致。A3/A5 使用训练标签计算重建损失权重，probe 设置不变。GCN 只用可见同切片表达构图，新细胞仅连接冻结训练参考库，不使用隐藏 XY 或查询之间的边。

若做短流程检查，统一改成一个 holdout、一个 seed、两轮训练，并将 `REUSE_A2_ROOT=None`，另用测试输出目录。正式运行恢复全部配置。


In [3]:
ARMS_TO_RUN = ['A0', 'A1']
DATA_GLOB = str(PROJECT.parent / "data/merfish_mouse_hypothalamus/**/merfish_[0-4].h5ad")
OUTPUT_ROOT = PROJECT / "outputs/euclidean_v1"
HOLDOUTS = [1, 2, 3]
SEEDS = [42, 0, 1, 2, 3]
EPOCHS = 100
DIM = 32
HIDDEN = 128
GRAPH_K = 10
BALANCE_POWER = 0.5  # 逆平方根频率，样本权重全训练集均值为 1。
LABEL_KEY = "cell_class"
LAYER = None
TRANSFORM = "log1p"  # X 已 log / 标准化时统一改 identity。
DEVICE = "cuda"
# 可复用上一轮兼容的 E0 AE；设为 None 则重新训练 A2。
REUSE_A2_ROOT = PROJECT / "outputs/representations_v2/E0_ae"
if not REUSE_A2_ROOT.is_dir():
    REUSE_A2_ROOT = None
CONFIG = deepcopy(EUCLIDEAN_CONFIG)
CONFIG.update(input_glob=DATA_GLOB, output_root=str(OUTPUT_ROOT), holdouts=HOLDOUTS,
              seeds=SEEDS, epochs=EPOCHS, dim=DIM, hidden=HIDDEN, graph_k=GRAPH_K,
              balance_power=BALANCE_POWER, label_key=LABEL_KEY, layer=LAYER,
              transform=TRANSFORM, device=DEVICE,
              reuse_a2_root=str(REUSE_A2_ROOT) if REUSE_A2_ROOT else None)
CONFIG


{'input_glob': '/3250608056/code/DeepSpatial/data/merfish_mouse_hypothalamus/**/merfish_[0-4].h5ad',
 'output_root': '/3250608056/code/DeepSpatial/MMDeepSpatial/outputs/euclidean_v1',
 'holdouts': [1, 2, 3],
 'seeds': [42, 0, 1, 2, 3],
 'label_key': 'cell_class',
 'layer': None,
 'transform': 'log1p',
 'n_genes': 0,
 'rare_fraction': 0.05,
 'dim': 32,
 'hidden': 128,
 'epochs': 100,
 'batch_size': 512,
 'lr': 0.001,
 'weight_decay': 1e-05,
 'grad_clip': 5.0,
 'curvature': 1.0,
 'fusion_alpha': 0.5,
 'log_every': 10,
 'e0_method': 'ae',
 'device': 'cuda',
 'cpu_threads': 4,
 'retrieval_ks': [1, 5, 15],
 'query_chunk': 128,
 'reference_chunk': 2048,
 'coverage_quantile': 0.95,
 'coverage_calibration_max': 2048,
 'probe_c': 1.0,
 'probe_class_weight': 'balanced',
 'probe_max_iter': 2000,
 'shuffled_probe': False,
 'max_tangent_radius': 1.5,
 'diagnostics': False,
 'diagnostic_alphas': [0.25, 0.75],
 'graph_k': 10,
 'graph_query_chunk': 512,
 'balance_power': 0.5,
 'reuse_a2_root': '/32506

In [4]:
paths = input_paths(CONFIG["input_glob"])
display(pd.DataFrame({"slice_index": range(len(paths)), "file": [str(p) for p in paths],
                      "held_out": [i in HOLDOUTS for i in range(len(paths))]}))
print("Arms:", ARMS_TO_RUN, "runs per arm:", len(HOLDOUTS)*len(SEEDS))
print("A0/A1 不训练神经编码器；A2 可复用已核验的旧权重。")


,slice_index,file,held_out
0,0,/3250608056/code/DeepSpatial/data/merfish_mous...,False
1,1,/3250608056/code/DeepSpatial/data/merfish_mous...,True
2,2,/3250608056/code/DeepSpatial/data/merfish_mous...,True
3,3,/3250608056/code/DeepSpatial/data/merfish_mous...,True
4,4,/3250608056/code/DeepSpatial/data/merfish_mous...,False


Arms: ['A0', 'A1'] runs per arm: 15
A0/A1 不训练神经编码器；A2 可复用已核验的旧权重。


In [5]:
results = pd.concat([run_arm(arm, CONFIG) for arm in ARMS_TO_RUN], ignore_index=True)
display(results[["arm", "holdout", "seed", "encoder_uses_labels", "encoder_reused",
                 "rare_macro_auprc", "ultra_rare_macro_auprc", "low_rare_macro_auprc",
                 "rare_macro_f1", "rare_macro_retrieval_hit_at_5", "common_macro_f1",
                 "trainable_parameters"]])


Start A0 holdout=1 seed=42 train=(22760, 155) method=raw balanced=False
 rare_macro_auprc  ultra_rare_macro_auprc  rare_macro_f1  overall_accuracy
         0.785706                0.634146       0.721496          0.927299
Start A0 holdout=1 seed=0 train=(22760, 155) method=raw balanced=False
 rare_macro_auprc  ultra_rare_macro_auprc  rare_macro_f1  overall_accuracy
         0.785706                0.634146       0.721496          0.927299
Start A0 holdout=1 seed=1 train=(22760, 155) method=raw balanced=False
 rare_macro_auprc  ultra_rare_macro_auprc  rare_macro_f1  overall_accuracy
         0.785706                0.634146       0.721496          0.927299
Start A0 holdout=1 seed=2 train=(22760, 155) method=raw balanced=False
 rare_macro_auprc  ultra_rare_macro_auprc  rare_macro_f1  overall_accuracy
         0.785706                0.634146       0.721496          0.927299
Start A0 holdout=1 seed=3 train=(22760, 155) method=raw balanced=False
 rare_macro_auprc  ultra_rare_macro_auprc  r

,arm,holdout,seed,encoder_uses_labels,encoder_reused,rare_macro_auprc,ultra_rare_macro_auprc,low_rare_macro_auprc,rare_macro_f1,rare_macro_retrieval_hit_at_5,common_macro_f1,trainable_parameters
0,A0,1,42,False,False,0.785706,0.634146,0.967579,0.721496,0.774663,0.939172,0
1,A0,1,0,False,False,0.785706,0.634146,0.967579,0.721496,0.774663,0.939172,0
2,A0,1,1,False,False,0.785706,0.634146,0.967579,0.721496,0.774663,0.939172,0
3,A0,1,2,False,False,0.785706,0.634146,0.967579,0.721496,0.774663,0.939172,0
4,A0,1,3,False,False,0.785706,0.634146,0.967579,0.721496,0.774663,0.939172,0
5,A0,2,42,False,False,0.807153,0.677558,0.962667,0.687972,0.828196,0.938190,0
6,A0,2,0,False,False,0.807153,0.677558,0.962667,0.687972,0.828196,0.938190,0
7,A0,2,1,False,False,0.807153,0.677558,0.962667,0.687972,0.828196,0.938190,0
8,A0,2,2,False,False,0.807153,0.677558,0.962667,0.687972,0.828196,0.938190,0
9,A0,2,3,False,False,0.807153,0.677558,0.962667,0.687972,0.828196,0.938190,0


In [6]:
metrics = ["rare_macro_auprc", "ultra_rare_macro_auprc", "low_rare_macro_auprc",
           "rare_macro_f1", "rare_macro_recall", "rare_macro_retrieval_hit_at_5",
           "rare_macro_knn_purity_at_15", "common_macro_f1"]
by_fold = results.groupby(["arm", "holdout"])[metrics].mean()
display(by_fold)
# 描述性切片间统计，不是独立生物学重复的置信区间。
display(by_fold.groupby("arm").agg(["mean", "std"]))


rare_macro_auprc  ultra_rare_macro_auprc  low_rare_macro_auprc  \
arm holdout                                                                   
A0  1                0.785706                0.634146              0.967579   
    2                0.807153                0.677558              0.962667   
    3                0.774421                0.615618              0.964985   
A1  1                0.688836                0.459073              0.964551   
    2                0.708098                0.503086              0.954111   
    3                0.695508                0.477508              0.957107   

             rare_macro_f1  rare_macro_recall  rare_macro_retrieval_hit_at_5  \
arm holdout                                                                    
A0  1             0.721496           0.754054                       0.774663   
    2             0.687972           0.755132                       0.828196   
    3             0.697724           0.750943                       0.825787   
A1  1             0.622107           0.725300                       0.747712   
    2             0.605449           0.791569                       0.796763   
    3             0.620727           0.786130                       0.754348   

             rare_macro_knn_purity_at_15  common_macro_f1  
arm holdout                                                
A0  1                           0.615487         0.939172  
    2                           0.613396         0.938190  
    3                           0.618891         0.938666  
A1  1                           0.572049         0.928044  
    2                           0.580872         0.915673  
    3                           0.597526         0.926981

rare_macro_auprc           ultra_rare_macro_auprc            \
                mean       std                   mean       std   
arm                                                               
A0          0.789093  0.016626               0.642441  0.031792   
A1          0.697480  0.009781               0.479889  0.022103   

    low_rare_macro_auprc           rare_macro_f1           rare_macro_recall  \
                    mean       std          mean       std              mean   
arm                                                                            
A0              0.965077  0.002457      0.702397  0.017244          0.753376   
A1              0.958590  0.005375      0.616095  0.009245          0.767666   

              rare_macro_retrieval_hit_at_5            \
          std                          mean       std   
arm                                                     
A0   0.002175                      0.809549  0.030236   
A1   0.036791                      0.766275  0.026612   

    rare_macro_knn_purity_at_15           common_macro_f1            
                           mean       std            mean       std  
arm                                                                  
A0                     0.615925  0.002773        0.938676  0.000491  
A1                     0.583482  0.012938        0.923566  0.006856

## 六组全部完成后统一比较

任何一个 notebook 将下方开关改为 True 后运行。比较会核对完整 seed/split、输入、预处理、源码、容量和 A4/A5 图一致性。保存六组原始指标、稀有度分层、逐类表及预定义配对差值，不自动选择最佳模型。


In [7]:
COMPARE_WHEN_READY = False
if COMPARE_WHEN_READY:
    all_runs, aggregate, all_types = compare_arms(OUTPUT_ROOT)
    display(aggregate)
    display(pd.read_csv(OUTPUT_ROOT / "comparison/paired_deltas.csv"))
